# E1: a saturating readout

Summing raw measured singles extrapolates straight past the assay range. On
TrpB3A the mean signed residual is the whole of its error, because three
deleterious singles sum to roughly -4 while the real triple sits at the
detection floor near -1.4.

Additive ridge wins on error only because L2 shrinkage happens to imitate that
saturation. Fitting the readout does the same job for the stated reason, and it
leaves a landscape with real topology behind.

Runs compared here:

- `output_260819_gate` sums raw singles.
- `output_260819_e1` composes through a fitted readout, prior measured on raw fitness.
- `output_260819_e1b` also measures the prior in the same latent space it is applied.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
from mutascape.benchmarks.reporting import (
    coverage_table,
    load_runs,
    median_table,
    paired_comparison,
    versus_baseline,
)

pd.set_option("display.width", 160)
ROOT = Path.cwd().parents[1] / "notebooks" / "iclr"

In [2]:
runs = load_runs(
    {
        "raw_singles": ROOT / "output_260819_gate",
        "readout": ROOT / "output_260819_e1",
        "latent_prior": ROOT / "output_260819_e1b",
        "final": ROOT / "output_260819_e1c",
    }
)

## Turning the readout on

Paired per landscape, so differing assay units cannot dominate.

In [3]:
paired_comparison(runs, "raw_singles", "readout").round(3)

,raw_singles,readout,median_delta,improved,n
metric,,,,,
spearman,0.247,0.244,-0.001,13,29
ndcg,0.925,0.922,-0.000,13,29
mae,0.854,0.314,-0.466,25,29
rmse,0.982,0.404,-0.447,25,29
wasserstein,0.791,0.264,-0.459,25,29
crps,0.610,0.263,-0.303,24,29
c2st_auc,0.911,0.889,-0.006,17,29
mmd_squared,0.347,0.212,-0.069,19,29


## Measuring the prior in the space it is applied

Tested and rejected. A readout compresses near its floor, so the leftover latent
residual is genuinely small and the intervals narrow further. Under-coverage is
not a space mismatch.

In [4]:
paired_comparison(runs, "readout", "latent_prior").round(3)

,readout,latent_prior,median_delta,improved,n
metric,,,,,
spearman,0.244,0.252,-0.001,13,29
ndcg,0.922,0.922,0.001,18,29
mae,0.314,0.342,0.014,4,29
rmse,0.404,0.461,0.012,4,29
wasserstein,0.264,0.287,0.013,3,29
crps,0.263,0.308,0.006,7,29
c2st_auc,0.889,0.906,0.011,8,29
mmd_squared,0.212,0.212,0.006,9,29


## Restoring the between-protein variance

Shrinking the depth location says the pooled shift does not transfer. It does not
say the target sits at the pooled mean, so the spread that justified the shrinkage
belongs in the predictive scale. Without it, sampled pair interactions are the
only source of uncertainty and every interval falls short.

In [5]:
paired_comparison(runs, "readout", "final").round(3)

,readout,final,median_delta,improved,n
metric,,,,,
spearman,0.244,0.246,-0.001,12,29
ndcg,0.922,0.920,-0.000,11,29
mae,0.314,0.301,-0.004,15,29
rmse,0.404,0.376,0.000,13,29
wasserstein,0.264,0.234,-0.007,15,29
crps,0.263,0.265,0.000,12,29
c2st_auc,0.889,0.899,-0.001,16,29
mmd_squared,0.212,0.202,0.000,12,29


## Against additive ridge

`wins` is the paired count across landscapes and is the honest comparison. The
medians on MAE and RMSE still favour ridge because a few wide-unit assays drag
them, which is why the protocol reports per-protein panels.

In [6]:
versus_baseline(runs["final"]).round(3)

,ridge_additive,posterior,wins,n
metric,,,,
spearman,0.238,0.246,21,29
ndcg,0.889,0.920,19,29
mae,0.256,0.301,16,29
rmse,0.319,0.376,18,29
wasserstein,0.254,0.234,17,29


## Calibration

Improved but still short of nominal. The model's only source of predictive spread
is sampled pair interactions plus the between-protein term. Readout-fit
uncertainty and higher-order structure are not represented yet.

In [7]:
coverage_table(runs).round(3)

raw_singles             readout            latent_prior               final           
           coverage mean_width coverage mean_width     coverage mean_width coverage mean_width
nominal                                                                                       
0.50          0.150      0.550    0.165      0.243         0.13      0.216    0.245      0.362
0.80          0.335      1.080    0.365      0.512         0.25      0.413    0.570      0.664
0.95          0.620      1.741    0.600      0.876         0.45      0.612    0.825      0.958

## Median metrics across every configuration

In [8]:
median_table(runs).round(3)

,raw_singles,readout,latent_prior,final
metric,,,,
spearman,0.247,0.244,0.252,0.246
ndcg,0.925,0.922,0.922,0.920
mae,0.854,0.314,0.342,0.301
rmse,0.982,0.404,0.461,0.376
wasserstein,0.791,0.264,0.287,0.234
crps,0.610,0.263,0.308,0.265
c2st_auc,0.911,0.889,0.906,0.899
mmd_squared,0.347,0.212,0.212,0.202
